# Notebook esample to convert an AMASS mocap/frame to Human3.6M

In [ ]:
import sys
if '../' not in sys.path:
    sys.path.append('../')


import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
import copy
from hmp_utils import ForwardKinematics, RotConvert
from hmp_utils.visualize.static_3d_plot import (
    interactive_plot_body_keypoints,
    plot_body_keypoints,
)
from hmp_utils import H36MConstants, SmplConstants, BodyModelType
from hmp_utils import MocapLoader, Amass2HumanConvert
from hmp_utils.visualize.stickman_animation import (
    pose_animation,
    CameraOrientation,
    AnimationMode,
)
from hmp_utils.visualize.static_3d_plot import plot_body_keypoints

### Load AMASS mocap

In [ ]:
filepath = './example_mocaps/walking_poses.npz'

# load mocap
mocap_loader=MocapLoader(body_model_type=SmplConstants.BODY_MODEL_TYPE,
                         keypoint_ids_to_load=SmplConstants.KEYPOINTS)
data = mocap_loader.load_mocap(filepath)
assert data is not None

# framerate
framerate = data.framerate

# select a specific frame
frame = 17

# get amass rotations and root translations
amass_motion_full = data.rots
amass_trans_full = data.trans
amass_motion_frame = amass_motion_full[frame]
amass_trans_frame = amass_trans_full[frame]
# amass_motion = np.zeros((24,3))

### Convert AMASS mocap to Human3.6M

convert and visualize new Human3.6M mocap

In [ ]:
# conveter from AMASS to human
converter=Amass2HumanConvert()
human_motion_full = converter.amass_to_h36m_mocap_fast(amass_motion_full, amass_trans_full)

# compute kinematics
kine = ForwardKinematics(H36MConstants.OFFSETS, H36MConstants.PARENTS)
kine.set_body_model_type(BodyModelType.H36M)
keypoints_positions = kine.compute_fk(human_motion_full[:,1:,:])/1000

# visualize converted mocap
motions_list = [keypoints_positions]
subplot_ids = [0]
subplot_colors = ['b']
subplot_titles = ['Human36m']
camera_orientations = CameraOrientation(-90, 90)
pose_animation(motions_list, parents=H36MConstants.PARENTS, animation_framerate=framerate, subplot_ids=subplot_ids, 
               camera_orientations=camera_orientations, subplot_titles=subplot_titles,
               subplot_colors=subplot_colors, animation_mode=AnimationMode.HTML,
               fig_opts={'dpi':100})

visualiza original AMASS mocap for comparison

In [ ]:
kine = ForwardKinematics(SmplConstants.OFFSETS, SmplConstants.PARENTS)
kine.set_body_model_type(BodyModelType.SMPL)
keypoints_positions = kine.compute_fk(amass_motion_full)

motions_list = [keypoints_positions]
subplot_ids = [0]
subplot_colors = ['b']
subplot_titles = ['Smpl']
camera_orientations = CameraOrientation(-90, 90)
pose_animation(motions_list, parents=SmplConstants.PARENTS, animation_framerate=framerate, subplot_ids=subplot_ids, 
               camera_orientations=camera_orientations, subplot_titles=subplot_titles,
               subplot_colors=subplot_colors, animation_mode=AnimationMode.HTML,
               fig_opts={'dpi':100})